In [1]:
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
uri = "bolt://localhost:7687"
username = "neo4j"
import os
password = os.environ["NEO4J_PASSWORD"]

driver = GraphDatabase.driver(uri, auth=(username, password))

In [3]:
def extract_node_properties_and_relationships():
    with driver.session() as session:
        # Cypher query to fetch node properties and relationships
        query = """
        MATCH (n)-[r]->(m)
            WHERE any(label IN labels(n) WHERE label IN ['Gejala', 'PenyakitPadi', 'HamaPadi']) 
            AND any(label IN labels(m) WHERE label IN ['Gejala', 'PenyakitPadi', 'HamaPadi'])
        RETURN 
         n.label AS target_label, labels(n) AS target_class, 
        type(r) AS relationship_type,
         m.label AS source_label, labels(m) AS source_class

        """
        result = session.run(query)
        # Extract properties and relationships and store in DataFrame
        df = pd.DataFrame([record.values() for record in result], columns=result.keys())
        # Clean labels
        df['source_class'] = df['source_class'].apply(lambda entry: [item for item in entry if item not in ["Resource", "NamedIndividual"]])
        df['target_class'] = df['target_class'].apply(lambda entry: [item for item in entry if item not in ["Resource", "NamedIndividual"]])
        return df

# Call the function to extract node properties and relationships
Rice_KG_df = extract_node_properties_and_relationships()

# Print the DataFrame
Rice_KG_df

ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)

In [ ]:
G = nx.DiGraph()
edges = Rice_KG_df[['source_label', 'target_label']].values.tolist()
G.add_edges_from(edges)

node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1, seed=42)
n2v_model = node2vec.fit(window=5, min_count=1, batch_words=4)

nodes = list(G.nodes())
node2idx = {node: idx for idx, node in enumerate(nodes)}
x = torch.tensor([n2v_model.wv[node] for node in nodes], dtype=torch.float)

edge_index = Rice_KG_df[['source_label', 'target_label']].applymap(lambda x: node2idx[x]).values.T
edge_index = torch.tensor(edge_index, dtype=torch.long)

Computing transition probabilities:   0%|          | 0/84 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 50/50 [00:00<00:00, 4038.89it/s]

C:\Users\Ariful\AppData\Local\Temp\ipykernel_26040\3321090964.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  edge_index = Rice_KG_df[['source_label', 'target_label']].applymap(lambda x: node2idx[x]).values.T
C:\Users\Ariful\AppData\Local\Temp\ipykernel_26040\3321090964.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  edge_index = Rice_KG_df[['source_label', 'target_label']].applymap(lambda x: node2idx[x]).values.T


In [ ]:
def is_gejala(label):
    kelas = Rice_KG_df.loc[Rice_KG_df['source_label'] == label, 'source_class']
    if kelas.empty:
        kelas = Rice_KG_df.loc[Rice_KG_df['target_label'] == label, 'target_class']
    kelas = kelas.iloc[0] if not kelas.empty else []
    return 'Gejala' in kelas

def is_hama_penyakit(label):
    kelas = Rice_KG_df.loc[Rice_KG_df['source_label'] == label, 'source_class']
    if kelas.empty:
        kelas = Rice_KG_df.loc[Rice_KG_df['target_label'] == label, 'target_class']
    kelas = kelas.iloc[0] if not kelas.empty else []
    return all(k in ['HamaPadi', 'PenyakitPadi'] for k in kelas)  # Menggunakan logika AND

# Label: 0 = bukan hama/penyakit, 1 = hama/penyakit
y = torch.tensor([1 if is_hama_penyakit(node) else 0 for node in nodes], dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

In [ ]:
class SimpleGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(SimpleGCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 2)  # 2 classes: 0 and 1

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
# Label: 0 = bukan hama/penyakit, 1 = hama/penyakit
y = torch.tensor([1 if is_hama_penyakit(node) else 0 for node in nodes], dtype=torch.long)

# Cross-validation evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []
precisions = []
recalls = []
f1s = []

In [ ]:
# Cross-validation evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []
precisions = []
recalls = []
f1s = []

for fold, (train_idx, test_idx) in enumerate(skf.split(range(len(nodes)), y.numpy())):
    print(f"Fold {fold + 1}:")
    
    # Create model for each fold
    model = SimpleGCN(in_channels=x.shape[1], hidden_channels=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    # Training loop
    for epoch in range(100):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = F.cross_entropy(out[train_idx], data.y[train_idx])
        loss.backward()
        optimizer.step()
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        pred = model(data).argmax(dim=1)
        y_true = data.y[test_idx].numpy()
        y_pred = pred[test_idx].numpy()
    
    # Calculate metrics for this fold
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print()

# Average metrics across folds
avg_accuracy = np.mean(accuracies)
avg_precision = np.mean(precisions)
avg_recall = np.mean(recalls)
avg_f1 = np.mean(f1s)

print("Cross-Validation Results:")
print(f"Average Accuracy: {avg_accuracy:.4f} ± {np.std(accuracies):.4f}")
print(f"Average Precision: {avg_precision:.4f} ± {np.std(precisions):.4f}")
print(f"Average Recall: {avg_recall:.4f} ± {np.std(recalls):.4f}")
print(f"Average F1-Score: {avg_f1:.4f} ± {np.std(f1s):.4f}")

Fold 1:
  Accuracy: 0.8235
  Precision: 0.8292
  Recall: 0.8235
  F1-Score: 0.8235

Fold 2:
  Accuracy: 0.8235
  Precision: 0.8269
  Recall: 0.8235
  F1-Score: 0.8223

Fold 3:
  Accuracy: 0.9412
  Precision: 0.9471
  Recall: 0.9412
  F1-Score: 0.9408

Fold 4:
  Accuracy: 0.8235
  Precision: 0.8269
  Recall: 0.8235
  F1-Score: 0.8223

Fold 3:
  Accuracy: 0.9412
  Precision: 0.9471
  Recall: 0.9412
  F1-Score: 0.9408

Fold 4:
  Accuracy: 0.9412
  Precision: 0.9471
  Recall: 0.9412
  F1-Score: 0.9408

Fold 5:
  Accuracy: 0.5000
  Precision: 0.2500
  Recall: 0.5000
  F1-Score: 0.3333

Cross-Validation Results:
Average Accuracy: 0.8059 ± 0.1617
Average Precision: 0.7601 ± 0.2605
Average Recall: 0.8059 ± 0.1617
Average F1-Score: 0.7721 ± 0.2256
  Accuracy: 0.9412
  Precision: 0.9471
  Recall: 0.9412
  F1-Score: 0.9408

Fold 5:
  Accuracy: 0.5000
  Precision: 0.2500
  Recall: 0.5000
  F1-Score: 0.3333

Cross-Validation Results:
Average Accuracy: 0.8059 ± 0.1617
Average Precision: 0.7601 ± 0.2

c:\Users\Ariful\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# Final model training on full data for prediction
model = SimpleGCN(in_channels=x.shape[1], hidden_channels=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.cross_entropy(out, data.y)
    loss.backward()
    optimizer.step()

model.eval()
pred = model(data).argmax(dim=1)

result_df = pd.DataFrame({'Node': nodes, 'Predicted_Label': pred.tolist(), 'Is_Gejala': [is_gejala(n) for n in nodes]})

gejala_nodes = result_df[result_df['Is_Gejala']]
predicted_hama_penyakit = result_df[(result_df['Predicted_Label'] == 1) & (~result_df['Is_Gejala'])]

In [ ]:
# Tampilkan daftar gejala dengan index
daftar_gejala = [n for n in nodes if is_gejala(n)]
print('Daftar gejala yang tersedia:')
for i, g in enumerate(daftar_gejala):
    print(f'{i}. {g}')

Daftar gejala yang tersedia:
0. Bulir terdapat bercak
1. Bulir mengalami kerusakan
2. Bulir pecah
3. Bulir berubah warna
4. Daun mengering
5. Malai terdapat bercak
6. Daun terdapat garis
7. Daun terdapat bercak
8. Daun menguning
9. Bibit mati
10. Batang membusuk
11. Tanaman tumbuh terhambat
12. Daun mengalami kerusakan
13. Daun rontok
14. Batang mengalami kerusakan
15. Daun berlubang
16. Tanaman terjadi pembentukan sporulasi
17. Malai mengalami kerusakan
18. Daun melepuh
19. Daun terdapat luka
20. Tanaman layu
21. Daun melengkung
22. Daun berkarat
23. Batang rapuh
24. Batang berlubang
25. Tanaman terdapat pupa
26. Malai hilang
27. Tanaman mengeluarkan cairan
28. Batang berubah warna
29. Bibit tumbuh tidak normal
30. Tanaman menghasilkan suara berbunyi
31. Tanaman rebah
32. Malai basah
33. Pelepah rapuh
34. Pelepah berubah warna
35. Pelepah mengalami kerusakan
36. Pelepah membusuk
37. Tanaman menurun produktivitas
38. Akar menghilang
39. Akar membusuk
40. Akar berlubang


In [ ]:
# Input index gejala dari pengguna (bisa lebih dari satu, pisahkan dengan koma)
input_idx = input('Masukkan index gejala (pisahkan dengan koma jika lebih dari satu): ').split(',')
input_idx = [int(idx.strip()) for idx in input_idx if idx.strip().isdigit() and int(idx.strip()) < len(daftar_gejala)]

# Cari node yang sesuai dengan index gejala
gejala_idx = [node2idx[daftar_gejala[i]] for i in input_idx]

if not gejala_idx:
    print('Index gejala tidak valid.')
else:
    # Cetak gejala yang diinputkan
    gejala_input = [daftar_gejala[i] for i in input_idx]
    print('Gejala yang diinputkan:', ', '.join(gejala_input))
    
    # Prediksi label untuk semua node
    model.eval()
    pred = model(data).argmax(dim=1)
    
    # Cari node hama/penyakit yang terhubung dengan semua gejala input
    hasil = set(nodes)
    for idx in gejala_idx:
        tetangga = set(G.successors(nodes[idx]))
        hasil = hasil.intersection(tetangga)  # Ambil irisan tetangga untuk semua gejala input
    
    # Filter hasil untuk node hama/penyakit
    hasil = [t for t in hasil if pred[node2idx[t]] == 1 and not is_gejala(t)]
    
    if hasil:
        print('Hama/Penyakit yang teridentifikasi dari gejala input:')
        for h in hasil:
            print('-', h)
    else:
        print('Tidak ditemukan hama/penyakit yang terhubung dengan semua gejala input.')

Gejala yang diinputkan: Batang membusuk, Bibit tumbuh tidak normal
Hama/Penyakit yang teridentifikasi dari gejala input:
- Bakanae


: 